# alpha-mipt-rag — прогон на Google Colab (A100 80 ГБ, vLLM)

Полный прогон RAG поверх корпуса `alfabank.ru` → отправляемый CSV, на GPU.

**Профиль:** embedder `bge-m3` (cuda) + reranker `bge-reranker-v2-m3` (cuda) + генератор через **vLLM** (батчами).
Дефолтный генератор — `Qwen/Qwen2.5-32B-Instruct-AWQ` (ungated, поддерживает system-роль, ~19 ГБ в 4-bit → много запаса на A100 80 ГБ).

**Порядок:** Runtime → Change runtime type → **A100 GPU**, затем выполняй ячейки сверху вниз.
Не используем `uv`/Python 3.13 — ставим зависимости через `pip` в штатный Python Colab.

> `google/gemma-3-27b-it`: она gated (нужен HF-токен с принятой лицензией) **и** её чат-шаблон не поддерживает system-роль — `rag/generation.py` придётся поправить (слить system в user). Подробнее — в ячейке выбора модели.

## 0. Проверка GPU
Убедись, что выдали именно A100 80 ГБ.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Параметры запуска

Заполни источники кода/данных и (опционально) HF-токен.

- `REPO_SRC` — откуда взять код. У репозитория нет git-remote, поэтому варианты:
  - `github`  → запушь репозиторий на GitHub и впиши URL в `GIT_URL`;
  - `drive`   → положи папку проекта на Google Drive и впиши путь в `DRIVE_REPO_DIR`;
  - `upload`  → загрузишь zip вручную (ячейка подскажет).
- Данные (`data/*.csv`, 28 МБ) и индекс **не в git** — `data/` берём из источника кода (если он его содержит) или с Drive.

In [ ]:
# --- источник кода ---
REPO_SRC = "drive"          # "github" | "drive" | "upload"
GIT_URL = "https://github.com/<you>/alpha_mipt_rag.git"   # если REPO_SRC=='github'
DRIVE_REPO_DIR = "/content/drive/MyDrive/alpha_mipt_rag"  # если REPO_SRC=='drive'
WORKDIR = "/content/alpha_mipt_rag"

# --- данные (если их нет внутри источника кода, укажи папку с websites.csv/questions.csv/sample_submission.csv) ---
DATA_SRC_DIR = ""          # пусто = данные уже лежат в WORKDIR/data; иначе путь, напр. /content/drive/MyDrive/alpha_data

# --- генератор ---
GEN_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"   # дефолт: ungated, system-роль OK, влезает с запасом
# GEN_MODEL = "google/gemma-3-27b-it"          # требует HF_TOKEN + правку generation.py (см. ниже)

# --- HF token (нужен только для gated-моделей вроде gemma) ---
HF_TOKEN = ""             # вставь токен или оставь пусто для ungated-модели

# --- сохранение результата на Drive, чтобы пережил рестарт сессии ---
USE_DRIVE = True
DRIVE_OUT_DIR = "/content/drive/MyDrive/alpha_mipt_rag_out"

print("REPO_SRC =", REPO_SRC, "| GEN_MODEL =", GEN_MODEL)

## 2. (опц.) Подключить Google Drive
Нужно, если код/данные на Drive или хочешь чекпоинтить результат туда.

In [ ]:
if REPO_SRC == "drive" or DATA_SRC_DIR.startswith("/content/drive") or USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Достать код

In [ ]:
import os, shutil, subprocess, sys

os.chdir("/content")   # важно: не оставаться внутри WORKDIR, иначе rmtree удалит CWD -> git clone exit 128

if REPO_SRC == "github":
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORKDIR], check=True)
elif REPO_SRC == "drive":
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    shutil.copytree(DRIVE_REPO_DIR, WORKDIR,
                    ignore=shutil.ignore_patterns(".venv", "__pycache__", ".git"))
elif REPO_SRC == "upload":
    from google.colab import files
    print("Загрузи zip репозитория...")
    up = files.upload()
    name = next(iter(up))
    os.makedirs(WORKDIR, exist_ok=True)
    shutil.unpack_archive(name, "/content/_unzip")
    # внутри zip обычно одна папка — найдём её
    inner = [p for p in os.listdir("/content/_unzip")]
    root = "/content/_unzip/" + inner[0] if len(inner) == 1 else "/content/_unzip"
    shutil.copytree(root, WORKDIR, dirs_exist_ok=True)

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print("WORKDIR:", os.getcwd())
print("files:", sorted(os.listdir(WORKDIR)))

In [ ]:
# Данные: если указан внешний источник — скопировать в data/
if DATA_SRC_DIR:
    os.makedirs("data", exist_ok=True)
    for fn in ["websites.csv", "questions.csv", "sample_submission.csv"]:
        src = os.path.join(DATA_SRC_DIR, fn)
        if os.path.exists(src):
            shutil.copy(src, os.path.join("data", fn))

for fn in ["websites.csv", "questions.csv", "sample_submission.csv"]:
    p = os.path.join("data", fn)
    assert os.path.exists(p), f"НЕТ {p} — укажи DATA_SRC_DIR"
    print(f"  data/{fn}: {os.path.getsize(p)/1024/1024:.1f} MB")

## 4. Зависимости

Ставим через `pip` (не `uv`). `vllm` тянет совместимый `torch`. faiss-cpu достаточно — индекс крошечный (7.5k векторов).

In [ ]:
# Зависимости. Колесо vllm на PyPI собрано под одну CUDA; если она не совпадает с torch в Colab,
# vllm._C падает с "libcudart.so.13: cannot open shared object file". uv с --torch-backend=auto
# подбирает torch+vllm под РЕАЛЬНУЮ CUDA этого рантайма. Ячейка ОДИН РАЗ перезапустит runtime —
# после рестарта нажми Runtime → Run all (второй проход пройдёт насквозь).
import subprocess, sys, os
MARK = "/content/.deps_done"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "pip", "install", "--python", sys.executable, "-q",
    "vllm>=0.9.0", "sentence-transformers>=3.0", "faiss-cpu",
    "polars>=1.0", "pydantic>=2.0", "pyyaml", "tqdm", "bm25s>=0.2",
    "--torch-backend=auto"], check=True)

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if not os.path.exists(MARK):
    open(MARK, "w").close()
    print("зависимости установлены — перезапускаю runtime, после рестарта: Runtime → Run all")
    os.kill(os.getpid(), 9)
else:
    import vllm
    print("OK: vllm", vllm.__version__)

In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "CUDA не доступна — проверь, что runtime = A100 GPU"

## 5. HF-токен (только для gated-моделей)

In [ ]:
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF login OK")
else:
    print("HF_TOKEN пуст — ок для ungated-модели", GEN_MODEL)

## 6. Привести config.yaml к GPU/vLLM-профилю

Чиним id генератора (в репо стоит несуществующий `gemma-4-27b`) и выставляем `cuda`/`vllm`.
Источник истины — `config.yaml`; патчим минимально, остальное берём из него.

In [ ]:
import yaml

with open("config.yaml", encoding="utf-8") as f:
    cfg_raw = yaml.safe_load(f)

cfg_raw["embedder"]["device"] = "cuda"
cfg_raw["reranker"]["device"] = "cuda"
cfg_raw["generator"]["backend"] = "vllm"
cfg_raw["generator"]["model"] = GEN_MODEL
cfg_raw["generator"]["quantization"] = None       # vLLM сам определит AWQ из конфига модели
cfg_raw["generator"]["gpu_memory_utilization"] = 0.90
cfg_raw["generator"]["tensor_parallel_size"] = 1

with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg_raw, f, allow_unicode=True, sort_keys=False)

print("generator.model =", cfg_raw["generator"]["model"])
print("embedder:", cfg_raw["embedder"]["model"], cfg_raw["embedder"]["device"])
print("reranker:", cfg_raw["reranker"]["model"], cfg_raw["reranker"]["device"])

## 7. Собрать индекс (chunks + FAISS + BM25)

Артефакты не в git → строим на месте (~3–5 мин: embed 7.5k чанков на GPU).

In [ ]:
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from rag.config import load_config, seed_everything
cfg = load_config()
seed_everything(cfg.seed)

needed = [cfg.resolve(cfg.paths.chunks_parquet),
          cfg.resolve(cfg.paths.faiss_index),
          cfg.resolve(cfg.paths.bm25_pickle)]
missing = [p for p in needed if not p.exists()]

if missing:
    print("строю индекс:", [p.name for p in missing])
    from rag.data import load_and_clean
    from rag.chunking import chunk_dataframe, save_chunks
    from rag.retrieval.dense import DenseRetriever
    from rag.retrieval.bm25 import BM25Retriever

    docs = load_and_clean(cfg.resolve(cfg.paths.websites_csv), cfg.cleaning)
    chunks_df = chunk_dataframe(docs, cfg.chunking)
    save_chunks(chunks_df, cfg.resolve(cfg.paths.chunks_parquet))

    db = DenseRetriever(cfg.embedder); db.build_index(chunks_df); db.save(cfg.resolve(cfg.paths.faiss_index))
    bb = BM25Retriever(); bb.build_index(chunks_df); bb.save(cfg.resolve(cfg.paths.bm25_pickle))
    del db, bb
    print("индекс готов")
else:
    print("индекс уже на месте:", [p.name for p in needed])

## 8. Загрузить модули пайплайна

Самое долгое — старт vLLM (скачивание весов генератора в первый раз + прогрев).

In [ ]:
import time
import polars as pl
from transformers import AutoTokenizer

from rag.chunking import load_chunks
from rag.retrieval.dense import DenseRetriever
from rag.retrieval.bm25 import BM25Retriever
from rag.rerank import Reranker
from rag.generation import Generator
from rag.pipeline import Pipeline
from rag.length import trim_answer
from rag.submission import write_submission
from rag.types import Answer

chunks = load_chunks(cfg.resolve(cfg.paths.chunks_parquet))
dense = DenseRetriever(cfg.embedder); dense.load(cfg.resolve(cfg.paths.faiss_index))
bm25 = BM25Retriever(); bm25.load(cfg.resolve(cfg.paths.bm25_pickle))
reranker = Reranker(cfg.reranker)
print(f"retrieval ready: {chunks.height} chunks, faiss dim={dense.dim}")

In [ ]:
t0 = time.time()
generator = Generator(cfg.generator, seed=cfg.seed)   # стартует vLLM + качает веса
tokenizer = AutoTokenizer.from_pretrained(cfg.chunking.tokenizer_model)
pipeline = Pipeline(cfg, chunks, dense, bm25, reranker, generator, tokenizer)
print(f"generator+pipeline ready in {time.time()-t0:.0f}s")

## 9. Дымовой тест

In [ ]:
ans = pipeline.answer("debug", "Как открыть карту Альфа-Банка?")
print(f"answer ({len(ans.answer)} chars):\n{ans.answer}")

## 10. Полный прогон — батчами + чекпоинт (resume)

Ключевое отличие от локального ноутбука: генерация идёт **батчами через vLLM** (а не по одному вопросу), а ответы пишутся в чекпоинт инкрементально. Если сессия Colab оборвётся — перезапусти ячейки 1–8, затем эту: она докатит только недостающие вопросы.

- `N_QUESTIONS = 20` — быстрая проверка; `None` — все ~6977.
- `BATCH` — размер батча генерации (на A100 80 ГБ можно 256–512).

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

N_QUESTIONS = None        # None = полный прогон
BATCH = 256

CKPT = Path(WORKDIR) / "submissions" / "checkpoint.csv"
CKPT.parent.mkdir(parents=True, exist_ok=True)
no_data = cfg.length.no_data_phrase

questions = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0)
if N_QUESTIONS is not None:
    questions = questions.head(N_QUESTIONS)

# resume: пропустить уже посчитанные q_id
done = set()
if CKPT.exists():
    done = set(pl.read_csv(CKPT, infer_schema_length=0)["q_id"].to_list())
rows = [r for r in questions.iter_rows(named=True) if str(r["q_id"]) not in done]
print(f"всего {questions.height}, уже сделано {len(done)}, осталось {len(rows)}")

def _append_ckpt(recs):
    df = pl.DataFrame({"q_id": [r[0] for r in recs], "answer": [r[1] for r in recs]})
    if CKPT.exists():
        with open(CKPT, "ab") as f:
            df.write_csv(f, include_header=False)
    else:
        df.write_csv(CKPT)

t0 = time.time()
for i in tqdm(range(0, len(rows), BATCH)):
    batch = rows[i:i + BATCH]
    # фаза 1: retrieval+rerank+grounding (per-question, но дёшево)
    meta = [(str(r["q_id"]), pipeline.retrieve_and_ground(str(r["q_id"]), r["query"])) for r in batch]
    # фаза 2: одна батч-генерация vLLM на все непустые контексты
    to_gen = [ctx for _, ctx in meta if ctx is not None]
    gen = iter(generator.generate_batch(to_gen)) if to_gen else iter(())
    recs = []
    for qid, ctx in meta:
        if ctx is None:
            recs.append((qid, no_data))
        else:
            raw = next(gen).strip() or no_data
            recs.append((qid, trim_answer(raw, cfg.length.answer_max_chars)))
    _append_ckpt(recs)

dt = time.time() - t0
print(f"готово: {len(rows)} вопросов за {dt:.0f}s ({dt/max(len(rows),1):.2f}s/q)")

## 11. Финальный submission + статистика длин

In [ ]:
# собрать финальный CSV из чекпоинта в точном формате sample_submission
ck = pl.read_csv(CKPT, infer_schema_length=0)
# порядок как в questions.csv
order = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0).select("q_id")
ck = order.join(ck, on="q_id", how="left")
answers = [Answer(q_id=str(r["q_id"]), answer=(r["answer"] or no_data)) for r in ck.iter_rows(named=True)]

OUT = cfg.resolve(cfg.paths.submission_csv)
write_submission(answers, sample_path=cfg.resolve(cfg.paths.sample_submission_csv), out_path=OUT)

out_df = pl.read_csv(OUT)
tcol = "answer_new" if "answer_new" in out_df.columns else "answer"
lens = out_df.with_columns(pl.col(tcol).str.len_chars().alias("L"))["L"]
nd = out_df.filter(pl.col(tcol) == no_data).height
print(f"rows={out_df.height}  chars: mean={lens.mean():.0f} p50={lens.quantile(0.5):.0f} p95={lens.quantile(0.95):.0f} max={lens.max()}")
print(f"no-data: {nd}/{out_df.height} ({100*nd/out_df.height:.1f}%)")

In [ ]:
# сохранить результат + чекпоинт на Drive, чтобы не потерять при закрытии сессии
if USE_DRIVE:
    Path(DRIVE_OUT_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT, os.path.join(DRIVE_OUT_DIR, "submission.csv"))
    shutil.copy(CKPT, os.path.join(DRIVE_OUT_DIR, "checkpoint.csv"))
    print("скопировано в", DRIVE_OUT_DIR)
else:
    from google.colab import files
    files.download(str(OUT))

## Перед отправкой (из CLAUDE.md, лимит 3 загрузки/день)

- [ ] Полный `questions.csv` прогнан без падений и пустых ответов.
- [ ] Распределение длин адекватно, хвостов ≥3× нет (см. статистику выше).
- [ ] no-data ответы выглядят разумно (не на очевидных вопросах).
- [ ] CSV совпал с `sample_submission.csv` по колонкам/UTF-8/разделителю (это гарантирует `write_submission`).
- [ ] Залогированы: `GEN_MODEL`, seed, версии моделей, `config.yaml`.
- [ ] Инференс локальный (vLLM на месте; скачивание весов — только setup).